#### Hybrid Retrieval Demo: Combining Sparse and Dense Approaches

- hybrid retrieval by combining sparse (BM25) and dense (OpenAI embedding) approaches on the same Wikipedia dataset.

#### Import Required Libraries

**Import all necessary libraries for sparse and dense retrieval, including numpy, pandas, rank_bm25, and OpenAI.**

In [1]:
# Install required packages if needed
# !pip install pandas numpy rank_bm25 faiss-cpu openai nltk

import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi
import faiss
from openai import OpenAI
import ast
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bhupe\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

#### Load Dataset

**Load the same Wikipedia dataset as used in the dense retrieval demo.**

In [2]:
# Path to the Wikipedia CSV file (update if needed)
data_path = r'D:\AI-DATASETS\02-MISC-large\GenAI-LLMs\chromadb\data\vector_database_wikipedia_articles_embedded.csv'

# Load the dataset
df = pd.read_csv(data_path)
df.sample(5)

,id,url,title,text,title_vector,content_vector,vector_id
20051,76780,https://simple.wikipedia.org/wiki/Popstars,Popstars,Popstars is a reality show where a number of p...,"[-0.006992892362177372, -0.010689516551792622,...","[-0.012987906113266945, -0.013546239584684372,...",20051
16230,62370,https://simple.wikipedia.org/wiki/Ring%20Lardner,Ring Lardner,"Ringgold Wilmer Lardner (March 6, 1885 – Septe...","[-0.017350586131215096, -0.006028478499501944,...","[-0.005358513444662094, 0.0005508752074092627,...",16230
24394,94043,https://simple.wikipedia.org/wiki/Itanagar,Itanagar,Itanagar is the capital of the Indian state o...,"[-0.002243243856355548, 0.0021156196016818285,...","[0.010061565786600113, -0.0014504932332783937,...",24394
10163,35064,https://simple.wikipedia.org/wiki/Belluno,Belluno,"Belluno (, , ) is a comune and city in the Ven...","[-0.009954195469617844, 0.005192749202251434, ...","[0.0061785075813531876, -0.012827820144593716,...",10163
13368,49017,https://simple.wikipedia.org/wiki/%C3%89dith%2...,Édith Piaf,"Édith Piaf (aka. ""La Môme Piaf"") (December 19,...","[-0.01402188092470169, 0.0016678492538630962, ...","[-0.019001100212335587, 0.0031948182731866837,...",13368


#### Parse Precomputed Dense Embeddings

**Parse the 'content_vector' column from string to numpy arrays for dense retrieval.**

In [3]:
# Parse the 'content_vector' column from string to numpy array
content_vectors = df['content_vector'].apply(lambda x: np.array(ast.literal_eval(x), dtype=np.float32))
embeddings      = np.stack(content_vectors.values)

print('Embeddings shape:', embeddings.shape)

Embeddings shape: (25000, 1536)


#### Build Sparse Retrieval Index (BM25)

**Create a sparse retrieval index using BM25 on the dataset.**

In [12]:
# Tokenize the documents for BM25
corpus           = df['text'].astype(str).tolist()
tokenized_corpus = [word_tokenize(doc.lower()) for doc in corpus]

# Build BM25 index
bm25 = BM25Okapi(tokenized_corpus)

#### Build Dense Retrieval Index (FAISS)

**Build a FAISS index using the precomputed dense embeddings.**

In [13]:
# Build FAISS index for dense retrieval
dim = embeddings.shape[1]

index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(f"FAISS index contains {index.ntotal} vectors.")

FAISS index contains 25000 vectors.


#### Hybrid Retrieval Function (Combine Sparse + Dense Scores)

**Implement a function that combines BM25 and dense (FAISS) scores for hybrid retrieval.**

In [14]:
# Get OpenAI embedding for a query
from openai import OpenAI
client = OpenAI()  

In [15]:
def get_query_embedding(query, model="text-embedding-3-small"):
    response = client.embeddings.create(input=query, model=model)
    return np.array(response.data[0].embedding, dtype=np.float32).reshape(1, -1)

In [16]:
# Hybrid retrieval function
def hybrid_retrieve(query, bm25, index, df, top_k=5, alpha=0.5):
    
    ##################################################
    # BM25 (sparse) scores
    tokenized_query = word_tokenize(query.lower())
    
    # Computes BM25 scores for all documents (higher = more keyword overlap).
    bm25_scores     = bm25.get_scores(tokenized_query)
    
    ##################################################
    # Dense (FAISS) scores
    
    # Gets the dense embedding for the query using OpenAI.
    query_emb = get_query_embedding(query)
    
    # Searches the FAISS index for all documents, returning distances (D) and indices (I).
    D, I      = index.search(query_emb, len(df))  # Get all distances
    
    # Initializes dense scores as zeros
    dense_scores       = np.zeros(len(df))
    
    # fills in the scores for the returned indices.
    dense_scores[I[0]] = -D[0]  # Negative L2 distance (higher is better) - why ???
    
    # Normalize scores
    bm25_norm  = (bm25_scores - np.min(bm25_scores))   / (np.ptp(bm25_scores) + 1e-8)
    dense_norm = (dense_scores - np.min(dense_scores)) / (np.ptp(dense_scores) + 1e-8)
    
    ##################################################
    # Combine (weighted sum)
    hybrid_score = alpha * bm25_norm + (1 - alpha) * dense_norm
    
    top_indices  = np.argsort(hybrid_score)[::-1][:top_k]
    
    results = df.iloc[top_indices].copy()
    
    results['bm25_score']   = bm25_scores[top_indices]
    results['dense_score']  = dense_scores[top_indices]
    results['hybrid_score'] = hybrid_score[top_indices]
    
    return results

#### Why use negative L2 distance for dense scores?

- FAISS returns L2 distances between vectors. A **smaller distance** means the vectors (and thus the texts) are **more similar**.
- For ranking, we want **higher scores to mean more relevant** ("higher is better"), just like BM25.
- By putting a **negative sign** in front of the distance (`-distance`), a smaller distance (more similar) becomes a bigger score.
- This way, all scores (BM25, dense, hybrid) follow the same rule: **higher = better match**.

**In short:**
- FAISS: smaller distance = more similar → we flip the sign so higher = better.
- BM25: higher score = more keyword overlap (already higher = better).
- Hybrid: combines both, so all scores are "higher is better" for easy comparison.

#### Examples

Suppose you have a query and three documents. FAISS returns these L2 distances:

| Document | L2 Distance | -L2 Distance (Dense Score) | Interpretation           |
|----------|-------------|---------------------------|--------------------------|
| Doc A    |    0.5      |         -0.5              | Most similar (best)      |
| Doc B    |    1.2      |         -1.2              | Less similar             |
| Doc C    |    2.0      |         -2.0              | Least similar (worst)    |

- **Without the negative sign:** Lower distance = better, but harder to compare with BM25 (where higher is better).
- **With the negative sign:** Higher dense score = better, now matches the "higher is better" logic of BM25 and hybrid scores.

----
**Guidelines for Interpreting Scores:**

- **bm25_score**: Measures keyword overlap between the query and document. **Higher is better** (more keyword matches).
- **dense_score**: Measures semantic similarity using vector distance. **Higher is better** (more semantically similar).
- **hybrid_score**: Weighted combination of bm25_score and dense_score (normalized). **Higher is better** (more relevant by both criteria).

**Tips:**
- Adjust the `alpha` parameter in the hybrid function to favor sparse (bm25) or dense (semantic) retrieval as needed.
- Use hybrid_score to rank results for best overall relevance in most RAG applications.
- Inspect all three scores to understand why a document was retrieved (e.g., high bm25 but low dense may mean keyword match but not semantic; high dense but low bm25 may mean semantic match but not exact words).

---
#### Run and Compare Hybrid, Sparse, and Dense Retrieval

**Try a sample query and compare the top results from hybrid, BM25-only, and dense-only retrieval.**

In [17]:
# Example query
query = "What is the capital of France?"

# Hybrid retrieval
hybrid_results = hybrid_retrieve(query, bm25, index, df, top_k=5, alpha=0.5)
print("Hybrid Retrieval Results:")

display_cols = ['id', 'title', 'hybrid_score', 'bm25_score', 'dense_score', 'text']
hybrid_results[display_cols]

Hybrid Retrieval Results:


,id,title,hybrid_score,bm25_score,dense_score,text
953,3829,Adverb,0.810170,24.333484,-1.884719,An adverb is a word used to tell more about a ...
1908,6362,Programming language,0.796349,21.816955,-1.870193,A programming language is a type of written la...
20409,78425,Longue paume,0.795504,23.302895,-1.882905,Longue paume is an outdoor version of jeu de p...
2412,7776,Madagascar,0.777367,23.711605,-1.894623,Madagascar is a large island nation in the Ind...
4317,13486,Supertramp,0.773679,21.647003,-1.879192,Supertramp is a British rock band. They were c...


In [18]:
# BM25-only retrieval
tokenized_query = word_tokenize(query.lower())

bm25_scores  = bm25.get_scores(tokenized_query)
bm25_top_idx = np.argsort(bm25_scores)[::-1][:5]
bm25_results = df.iloc[bm25_top_idx].copy()

bm25_results['bm25_score'] = bm25_scores[bm25_top_idx]

print("\nBM25-only Retrieval Results:")
bm25_results[['id', 'title', 'bm25_score', 'text']]


BM25-only Retrieval Results:


,id,title,bm25_score,text
14756,55636,Ecclesiology,27.679152,"In Christian theology, ecclesiology is the stu..."
306,590,Philosophy,27.530708,Philosophy is the study of underlying things. ...
4192,13149,Aesthetics,27.213347,Aesthetics is a branch of philosophy. It is th...
2119,7253,Epistemology,26.901634,Epistemology is the philosophy of knowledge. ...
777,3448,Question,26.743335,"A question is what someone asks, usually when ..."


In [19]:
# Dense-only retrieval
query_emb = get_query_embedding(query)

D, I = index.search(query_emb, 5)

dense_results = df.iloc[I[0]].copy()

dense_results['dense_score'] = -D[0]

print("\nDense-only Retrieval Results:")
dense_results[['id', 'title', 'dense_score', 'text']]


Dense-only Retrieval Results:


,id,title,dense_score,text
19369,73701,North American F-86 Sabre,-1.825412,"The F-86 Sabre (nicknamed the ""Sabre jet"") was..."
20447,78550,Republic P-47 Thunderbolt,-1.847538,The P-47 Thunderbolt (also called The Jug ) wa...
19523,74703,General Dynamics F-16 Fighting Falcon,-1.852056,The General Dynamics F-16 Fighting Falcon is a...
20147,77188,Grumman F4F Wildcat,-1.853292,The F4F Wildcat was a fighter aircraft made by...
3616,10981,Copyleft,-1.854643,Copyleft is a name for a type of a license for...
